In [0]:
data = [
    (1, "Alice", 1000),
    (2, "Bob", 2000),
    (3, "Charlie", 3000)
]

columns = ["id", "name", "salary"]

df = spark.createDataFrame(data, columns)
df.show()

In [0]:
df.write.format("delta").mode("overwrite").saveAsTable('workspace.default.people')

In [0]:
%sql
describe history workspace.default.people

In [0]:
df_uc = spark.read.table("workspace.default.people")
df_uc.show()

1️⃣ ACID Transactions

In [0]:
df.write.mode("append").saveAsTable("workspace.default.people")

In [0]:
%sql
RESTORE TABLE workspace.default.people TO VERSION AS OF 0

In [0]:
df_old = spark.read \
    .option("versionAsOf", 0) \
    .table("workspace.default.people")


df_old.show()

In [0]:
new_data = [(4, "David", 4000, "India")]
df_new = spark.createDataFrame(new_data, ["id", "name", "salary", "country"])

df_new.write.format("delta") \
    .mode("append") \
    .option("mergeSchema", "true") \
    .saveAsTable("workspace.default.people")

In [0]:
%sql
select * from workspace.default.people

In [0]:
new_data = [(4, "David", 4000, "India")]
df_new = spark.createDataFrame(new_data, ["id", "name", "salary", "country"])

df_new.write.format("delta") \
    .mode("overwrite") \
    .option("mergeSchema", "true") \
    .saveAsTable("workspace.default.people")

In [0]:
%sql
create or replace table workspace.default.people as select * from source_table

In [0]:
%sql
select  * from workspace.default.people

In [0]:
#incremental load

In [0]:
%sql
create table if not exists workspace.default.people
as
select * from  source_table_1



In [0]:
%sql
select max(version) from (desc history workspace.default.people)

In [0]:
from delta.tables import DeltaTable

delta_table = DeltaTable.forName(spark, "workspace.default.people")
df_updates=spark.read.table(source_table_1).filter("version">0)
delta_table.alias("target").merge(
    df_updates.alias("source"),
    "target.id = source.id"
).whenMatchedUpdate(set={
    "salary": "source.salary",
    "country":"source.country"
}).whenNotMatchedInsertAll().execute()

In [0]:
from delta.tables import DeltaTable

delta_table = DeltaTable.forName(spark, "workspace.default.people")

updates = [(2, "Bob", 5000,"India"),(8, "Bob1", 5000,"India")]
df_updates = spark.createDataFrame(updates, ["id", "name", "salary", "country"])

delta_table.alias("target").merge(
    df_updates.alias("source"),
    "target.id = source.id"
).whenMatchedUpdate(set={
    "salary": "source.salary",
    "country":"source.country"
}).whenNotMatchedInsertAll().execute()

In [0]:
spark.sql("""
ALTER TABLE workspace.default.people
SET TBLPROPERTIES (delta.enableChangeDataFeed = true)
""")

In [0]:
%sql
desc history workspace.default.people

In [0]:
from pyspark.sql.functions import col

In [0]:
df_cdc = spark.read.format("delta") \
    .option("readChangeFeed", "true") \
    .option("startingVersion", 11) \
    .table("workspace.default.people").filter(col("_change_type")=="insert")

df_cdc.show()

When you keep writing data (streaming / batch), Delta tables end up with:

📁 Thousands of small files    

🔍 Data randomly scattered

🐢 Queries become slow


In [0]:
%sql
optimize workspace.default.people zorder by (id,name)

In [0]:
%sql
SELECT * FROM workspace.default.people WHERE id = 4

In [0]:
spark.sql("OPTIMIZE workspace.default.people")

![](https://images.openai.com/static-rsc-4/FFWPiR3ZHTlyy5do5BUBQ9SKxH_5uvwC9Z-TdV8BOdW3aja79mZRXLNYx3yPER_TGf6smJVkvvoPCrYkticnHEJt2p948gmcz_qWdLX_YaST7dnlMGa72obVJ6c_n2UWRIezZa29EyshCaddDmyp-0q1A0x5ZHQbWp3IETATSkF4j8PlFOIjFKf_9l_QIkFX?purpose=fullsize)

✅ What happens:

Data is physically reordered

Rows with similar id values stored closer together

🔥 Real Impact (Most Important)
Without ZORDER:

id = 4 spread across many files ❌

Spark scans everything

With ZORDER:

id = 4 located in few files only ✅

Spark scans minimal data

In [0]:
%sql
select * from table where id=1

In [0]:
OPTIMIZE reduces small files to improve scan efficiency, while ZORDER clusters data to enable data skipping and faster query performance.

“Liquid Clustering is an adaptive clustering mechanism in Delta Lake that automatically organizes data based on query patterns, eliminating the need for manual OPTIMIZE and ZORDER.”

🧠 Why Liquid Clustering was introduced?
Problem with ZORDER:

  ❌ Manual (you must run OPTIMIZE)

  ❌ Expensive for large tables

  ❌ Static (fixed clustering once done)

✅ Liquid Clustering solves this:

  🔄 Automatic clustering

  ⚡ Incremental updates (only new data reorganized)

  🧠 Adaptive (adjusts as data grows)

  💰 Lower cost than repeated OPTIMIZE

In [0]:
👉 Deletion Vector (DV) = a way to logically delete rows without rewriting data files

Instead of:

❌ Rewriting entire Parquet files (slow, expensive)

Delta does:

✅ Marks rows as deleted using a bitmap (vector)

In [0]:
%sql
ALTER TABLE workspace.default.people
SET TBLPROPERTIES (
  'delta.enableDeletionVectors' = true
);